**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# 图像特征练习
*请完成并提交这份 worksheet，包括其中的输出以及 worksheet 外部的任何辅助代码。更多细节请参考课程网站上的 [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html)。*

我们已经看到，在图像分类任务中，直接在输入图像的像素上训练线性分类器可以达到还不错的性能。在这个练习中，我们将展示：如果不是在原始像素上训练线性分类器，而是在从原始像素计算出的特征上训练分类器，分类性能可以进一步提升。

本练习的所有工作都将在这个 notebook 中完成。

In [ ]:
import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt


%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# 用于自动重新加载外部模块
# 参考 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

## 加载数据
和前面的练习类似，我们将从磁盘加载 CIFAR-10 数据。

In [ ]:
from cs231n.features import color_histogram_hsv, hog_feature

def get_CIFAR10_data(num_training=49000, num_validation=1000, num_test=1000):
    # 加载原始 CIFAR-10 数据
    cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

    # 清理变量，避免多次加载数据导致内存问题
    try:
       del X_train, y_train
       del X_test, y_test
       print('Clear previously loaded data.')
    except:
       pass

    X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

    # 对数据做子采样
    mask = list(range(num_training, num_training + num_validation))
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = list(range(num_training))
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = list(range(num_test))
    X_test = X_test[mask]
    y_test = y_test[mask]

    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = get_CIFAR10_data()

## 提取特征
对每张图像，我们会计算方向梯度直方图（Histogram of Oriented Gradients，HOG），并使用 HSV 颜色空间中的 hue 通道计算颜色直方图。最终，我们通过拼接 HOG 特征向量和颜色直方图特征向量，为每张图像构造特征向量。

粗略来说，HOG 应该能捕捉图像纹理，同时忽略颜色信息；颜色直方图表示输入图像的颜色，同时忽略纹理。因此，我们预期同时使用二者会比只使用其中一种效果更好。你也可以出于兴趣自己验证这个假设。

`hog_feature` 和 `color_histogram_hsv` 函数都作用于单张图像，并返回该图像的特征向量。`extract_features` 函数接收一组图像和一组特征函数，对每张图像运行每个特征函数，并将结果存储在一个矩阵中；矩阵的每一列都是某一张图像所有特征向量拼接后的结果。

In [ ]:
from cs231n.features import *

# num_color_bins = 10 # 颜色直方图的 bin 数量
num_color_bins = 25 # 颜色直方图的 bin 数量
feature_fns = [hog_feature, lambda img: color_histogram_hsv(img, nbin=num_color_bins)]
X_train_feats = extract_features(X_train, feature_fns, verbose=True)
X_val_feats = extract_features(X_val, feature_fns)
X_test_feats = extract_features(X_test, feature_fns)

# 预处理：减去特征均值
mean_feat = np.mean(X_train_feats, axis=0, keepdims=True)
X_train_feats -= mean_feat
X_val_feats -= mean_feat
X_test_feats -= mean_feat

# 预处理：除以标准差，确保每个特征的尺度大致一致。
std_feat = np.std(X_train_feats, axis=0, keepdims=True)
X_train_feats /= std_feat
X_val_feats /= std_feat
X_test_feats /= std_feat

# 预处理：添加 bias 维度
X_train_feats = np.hstack([X_train_feats, np.ones((X_train_feats.shape[0], 1))])
X_val_feats = np.hstack([X_val_feats, np.ones((X_val_feats.shape[0], 1))])
X_test_feats = np.hstack([X_test_feats, np.ones((X_test_feats.shape[0], 1))])

## 在特征上训练 Softmax 分类器
使用本作业前面实现的 Softmax 代码，在上面提取的特征上训练 Softmax 分类器；这应该比直接在原始像素上训练得到更好的结果。

In [ ]:
# 使用验证集调优学习率和正则化强度。

from cs231n.classifiers.linear_classifier import Softmax

learning_rates = [1e-7, 1e-6]
regularization_strengths = [5e5, 5e6]

results = {}
best_val = -1
best_softmax = None

################################################################################
# TODO:                                                                        #
# 使用验证集设置学习率和正则化强度。                         #
# 这个流程应与 Softmax 中做过的验证相同；                  #
# 将训练得到的最佳分类器保存到 best_softmax。若调参足够仔细，#
# 你应该能在验证集上得到高于 0.42 的准确率。                 #
################################################################################


# 打印结果。
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved: %f' % best_val)

In [ ]:
# 在测试集上评估训练好的 Softmax；你应该至少能得到 0.42 的准确率。
y_test_pred = best_softmax.predict(X_test_feats)
test_accuracy = np.mean(y_test == y_test_pred)
print(test_accuracy)

In [ ]:
# 保存最佳 softmax 模型
best_softmax.save("best_softmax_features.npy")

In [ ]:
# 理解算法工作方式的一个重要方法，是可视化它犯下的错误。
# 在这个可视化中，我们展示一些被当前系统误分类的图像样本。
# 第一列展示系统预测为 “plane” 但真实标签不是 plane 的图像；
# 其他列依此类推。

examples_per_class = 8
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for cls, cls_name in enumerate(classes):
    idxs = np.where((y_test != cls) & (y_test_pred == cls))[0]
    idxs = np.random.choice(idxs, examples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt.subplot(examples_per_class, len(classes), i * len(classes) + cls + 1)
        plt.imshow(X_test[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls_name)
plt.show()

### 内联问题 1：
描述你看到的误分类结果。它们是否合理？


$\color{blue}{\textit 你的回答：}$

## 在图像特征上训练神经网络
在本作业前面，我们看到在原始像素上训练两层神经网络，比在原始像素上训练线性分类器有更好的分类性能。在这个 notebook 中，我们又看到在图像特征上训练线性分类器，优于在原始像素上训练线性分类器。

为了完整起见，我们还应该尝试在图像特征上训练神经网络。这种方法应该优于前面所有方法：你应该很容易在测试集上达到 55% 以上的分类准确率；我们的最佳模型大约达到 60% 的分类准确率。

In [ ]:
# 预处理：移除 bias 维度
# 确保这个单元只运行一次。
print(X_train_feats.shape)
X_train_feats = X_train_feats[:, :-1]
X_val_feats = X_val_feats[:, :-1]
X_test_feats = X_test_feats[:, :-1]

print(X_train_feats.shape)

In [ ]:
from cs231n.classifiers.fc_net import TwoLayerNet
from cs231n.solver import Solver

input_dim = X_train_feats.shape[1]
hidden_dim = 500
num_classes = 10

data = {
    'X_train': X_train_feats,
    'y_train': y_train,
    'X_val': X_val_feats,
    'y_val': y_val,
    'X_test': X_test_feats,
    'y_test': y_test,
}

net = TwoLayerNet(input_dim, hidden_dim, num_classes)
best_net = None

################################################################################
# TODO：在图像特征上训练一个两层神经网络。你可能需要调优超参数。       #
# 和前面几节一样，可以通过交叉验证调不同超参数。将最佳模型    #
# 存入 best_net 变量。                                           #
################################################################################


In [ ]:
# 在测试集上运行你最好的神经网络分类器。
# 目标是得到超过 58% 的准确率；细致调参也可以超过 60%。
#

y_test_pred = np.argmax(best_net.loss(data['X_test']), axis=1)
test_acc = (y_test_pred == data['y_test']).mean()
print(test_acc)

In [ ]:
# 保存最佳模型
best_net.save("best_two_layer_net_features.npy")